# Thử Nghiệm Mở Rộng: Level 7 (400 / 800 / 1000 / 1500 sims) vs Level 3 (4000 nodes)

**Mục tiêu:** Kiểm tra xem khi tăng ngân sách mô phỏng của `Level7_GuidedMCTS` lên `400, 800, 1000, 1500 sims/nước` (dùng chung mạng `checkpoints/best_model.pt` trên GPU), sức cờ của Level 7 có thu hẹp khoảng cách hoặc vượt qua `Level3_AlphaBeta (4000 nodes)` (đang thắng L7 `80.5%` ở mức 100 sims) hay không.

- **Kết quả được lưu tự động sau mỗi mức `N_sim`** vào `results/ablation2/test_lv7_more_sims.json` (hỗ trợ chạy tiếp nối nếu dừng giữa chừng).
- **Thời gian dự kiến (trên GPU GTX 1650/T4, `NUM_OPENINGS=10` = 20 ván/mức):** ~15–25 phút cho cả 4 mức.

In [ ]:
import json
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from chess_ai.agents.registry import create_agent
from chess_ai.arena.tournament import run_paired_matchup
from chess_ai.utils.epd import load_epd_file

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CUDA available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 1. Cấu Hình & Chạy Đối Đầu (`Level 7` vs `Level 3`)
Tăng `NUM_OPENINGS` (ví dụ `10` = 20 ván/mức, hoặc `25` = 50 ván/mức) tùy quỹ thời gian. Đặt `FORCE_RERUN = True` nếu muốn xóa kết quả cũ trong `test_lv7_more_sims.json` để chạy lại từ đầu.

In [ ]:
SIM_BUDGETS = [400, 800, 1000, 1500]
NUM_OPENINGS = 10      # 10 khai cuộc x 2 màu = 20 ván/mức (tăng lên 25 nếu muốn 50 ván/mức)
L3_MAX_NODES = 4000
MAX_PLIES = 160
SEEDS = [42]
FORCE_RERUN = False

OUT_JSON = PROJECT_ROOT / "results" / "ablation2" / "test_lv7_more_sims.json"
OUT_JSON.parent.mkdir(parents=True, exist_ok=True)

openings = load_epd_file(PROJECT_ROOT / "resources" / "openings_50.epd")[:NUM_OPENINGS]
l5_ckpt = str(PROJECT_ROOT / "checkpoints" / "best_model.pt")

saved = {"results": []} if FORCE_RERUN or not OUT_JSON.exists() else json.loads(OUT_JSON.read_text(encoding="utf-8"))
done_map = {int(r["simulations"]): r for r in saved.get("results", [])}

agent_l3 = create_agent("level3", default_depth=3, max_nodes=L3_MAX_NODES)

for n_sim in SIM_BUDGETS:
    if n_sim in done_map:
        print(f"[Skip] N_sim={n_sim} đã có sẵn trong {OUT_JSON.name} (Score={done_map[n_sim]['score']*100:.1f}%).")
        continue

    t0 = time.perf_counter()
    print(f"\n[Run] Level7_GuidedMCTS ({n_sim} sims) vs Level3_AlphaBeta ({L3_MAX_NODES} nodes) — {len(openings)*2} ván...", flush=True)
    agent_l7 = create_agent("level7", checkpoint_path=l5_ckpt, simulations=n_sim, eval_mode=True, seed=42)
    res = run_paired_matchup(
        agent_a=agent_l7,
        agent_b=agent_l3,
        openings=openings,
        seeds=SEEDS,
        budget_mode="workload_control",
        workload_a=n_sim,
        workload_b=L3_MAX_NODES,
        max_plies=MAX_PLIES,
        bootstrap_resamples=5000,
        workers=1,
    )
    res["simulations"] = n_sim
    res["elapsed_sec"] = round(time.perf_counter() - t0, 2)
    done_map[n_sim] = res

    payload = {
        "experiment": "Level7_More_Sims_vs_Level3",
        "l3_max_nodes": L3_MAX_NODES,
        "games_per_budget": len(openings) * 2,
        "sim_budgets": SIM_BUDGETS,
        "results": [done_map[s] for s in SIM_BUDGETS if s in done_map],
    }
    OUT_JSON.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"  -> Xong N_sim={n_sim}: Score L7={res['score']*100:.1f}% (W/D/L={res['wins']}/{res['draws']}/{res['losses']}) | L7 Avg={res['agent_a_avg_latency_ms']:.1f}ms | {res['elapsed_sec']}s")


## 2. Tổng Hợp Bảng Số Liệu & Biểu Đồ Phân Tích

In [ ]:
data = json.loads(OUT_JSON.read_text(encoding="utf-8"))
rows = data.get("results", [])
assert rows, "Chưa có dữ liệu trong test_lv7_more_sims.json!"

# Nạp thêm mốc 100 sims từ dual_mode_200 (M4_WL_L3_vs_L7: L3 thắng 80.5% => L7 đạt 19.5%)
baseline_100 = None
dm_path = PROJECT_ROOT / "results" / "dual_mode_200" / "dual_mode_200_summary.json"
if dm_path.exists():
    dm = json.loads(dm_path.read_text(encoding="utf-8"))
    m4 = next((m for m in dm.get("matchups", []) if m.get("matchup_id") == "M4_WL_L3_vs_L7"), None)
    if m4:
        baseline_100 = 100.0 - float(m4["score_pct"])

df = pd.DataFrame([
    {
        "N_sim (L7)": r["simulations"],
        "Số ván": r["games"],
        "Thắng (W)": r["wins"],
        "Hòa (D)": r["draws"],
        "Thua (L)": r["losses"],
        "Điểm S L7 (%)": round(r["score"] * 100.0, 1),
        "Trắng (%)": round(r["white_score"] * 100.0, 1),
        "Đen (%)": round(r["black_score"] * 100.0, 1),
        "Wilson 95% CI": f"[{r['wilson_ci_95'][0]*100:.1f}%, {r['wilson_ci_95'][1]*100:.1f}%]",
        "L7 Avg ms": round(r["agent_a_avg_latency_ms"], 1),
        "L3 Avg ms": round(r["agent_b_avg_latency_ms"], 1),
    }
    for r in rows
])
display(df)

xs = [r["simulations"] for r in rows]
scores = [r["score"] * 100.0 for r in rows]
ci_lo = [r["wilson_ci_95"][0] * 100.0 for r in rows]
ci_hi = [r["wilson_ci_95"][1] * 100.0 for r in rows]
l7_ms = [r["agent_a_avg_latency_ms"] for r in rows]
l3_ms = [r["agent_b_avg_latency_ms"] for r in rows]

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4), dpi=130)

# Biểu đồ 1: Điểm S (%) của L7 theo N_sim kèm Wilson 95% CI
ax = axes[0]
ax.plot(xs, scores, marker="o", linewidth=2.4, color="#16A34A", label="Level 7 Score (%)")
ax.fill_between(xs, ci_lo, ci_hi, color="#16A34A", alpha=0.18, label="Wilson 95% CI")
ax.axhline(50.0, color="#EF4444", linestyle="--", linewidth=1.3, label="Cân bằng L3 (50%)")
if baseline_100 is not None:
    ax.scatter([100], [baseline_100], color="#2563EB", zorder=5, s=60, label=f"Mốc 100 sims ({baseline_100:.1f}%)")
for x_v, y_v in zip(xs, scores):
    ax.text(x_v, y_v + 2.8, f"{y_v:.1f}%", ha="center", fontsize=8.5, fontweight="bold", color="#0F172A")
ax.set_title("Sức cờ Level 7 vs Level 3 theo N_sim", fontweight="bold")
ax.set_xlabel("Ngân sách mô phỏng L7 (N_sim)")
ax.set_ylabel("Điểm quy đổi S (%)")
ax.set_ylim(-2, 105)
ax.grid(True, linestyle="--", alpha=0.35)
ax.legend(fontsize=8)

# Biểu đồ 2: Phân bố Thắng / Hòa / Thua
ax = axes[1]
idx = np.arange(len(xs))
w = 0.25
ax.bar(idx - w, df["Thắng (W)"], width=w, color="#22C55E", label="Thắng (W)")
ax.bar(idx,     df["Hòa (D)"],   width=w, color="#F59E0B", label="Hòa (D)")
ax.bar(idx + w, df["Thua (L)"],  width=w, color="#EF4444", label="Thua (L)")
ax.set_xticks(idx); ax.set_xticklabels([str(x) for x in xs])
ax.set_title("Phân bố Thắng / Hòa / Thua của L7", fontweight="bold")
ax.set_xlabel("N_sim (L7)")
ax.set_ylabel("Số ván")
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.legend(fontsize=8)

# Biểu đồ 3: Đánh đổi Độ trễ TB mỗi nước (ms)
ax = axes[2]
ax.plot(xs, l7_ms, marker="s", linewidth=2.2, color="#2563EB", label="Level 7 Avg Latency (ms)")
ax.plot(xs, l3_ms, marker="^", linewidth=2.0, linestyle="--", color="#64748B", label=f"Level 3 ({data['l3_max_nodes']} nodes)")
for x_v, y_v in zip(xs, l7_ms):
    ax.text(x_v, y_v + max(l7_ms)*0.03, f"{y_v:.0f}ms", ha="center", fontsize=8, color="#1E293B")
ax.set_title("Độ trễ trung bình mỗi nước (ms)", fontweight="bold")
ax.set_xlabel("N_sim (L7)")
ax.set_ylabel("Thời gian / nước (ms)")
ax.grid(True, linestyle="--", alpha=0.35)
ax.legend(fontsize=8)

fig.suptitle("Thử nghiệm mở rộng: Level 7 (400–1500 sims) vs Level 3 (4000 nodes)", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()
